# Simple RAG Example with Weaviate and LangChain

## 1. Introduction

This notebook demonstrates a simple implementation of the Retrieval-Augmented Generation (RAG) pattern. The goal is to build a question-answering system that leverages a vector database to provide context-aware answers from a Large Language Model (LLM).

The process involves:
1.  **Data Preparation**: Creating a small, factual dataset of scientific notes.
2.  **Environment Setup**: Preparing the Docker environment and installing dependencies.
3.  **Database Deployment**: Launching a Weaviate vector database instance using Docker.
4.  **Embeddings & Ingestion**: Generating vector embeddings for our data using Azure OpenAI and loading it into Weaviate.
5.  **RAG Experiment**: Executing a RAG pipeline:
    - Expanding a user's question.
    - Searching for relevant documents in Weaviate.
    - Generating a final answer using an LLM augmented with the retrieved documents.
6.  **Cleanup**: Removing the Docker container to free up system resources.

## 2. System Environment Preparation

This section contains helper functions to interact with the underlying operating system (Linux or Windows with WSL) to manage Docker containers and file paths.

### 2.1. WSL and Shell Command Helpers

In [1]:
import platform
import subprocess
import os

# --- WSL Detection ---
system = platform.system()
USE_WSL = system == "Windows"
print(f"Operating System: {system}. Using WSL for Docker commands: {USE_WSL}")

# --- Shell Command Helpers ---
def run_wsl_command(command):
    """Executes a command inside WSL and returns the result."""
    result = subprocess.run(
        ["wsl", "-e", "bash", "-l", "-c", command],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )
    return {
        "returncode": result.returncode,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
        "success": result.returncode == 0
    }

def run_linux_command(command):
    """Executes a command in a standard Linux/macOS shell."""
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )
    return {
        "returncode": result.returncode,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
        "success": result.returncode == 0
    }

def run_shell_command(command):
    """Universal function to run a shell command, abstracting WSL usage."""
    if USE_WSL:
        return run_wsl_command(command)
    else:
        return run_linux_command(command)

print("✅ Shell command helpers are defined.")

Operating System: Linux. Using WSL for Docker commands: False
✅ Shell command helpers are defined.


### 2.2. Install Dependencies

In [2]:
import sys

!"{sys.executable}" -m pip install -q weaviate-client==4.18.1 langchain~=0.3.0 langchain-openai~=0.2.0 python-dotenv~=1.0.0 pandas~=2.2.0

print("✅ Required libraries have been installed.")

✅ Required libraries have been installed.


In [3]:
!"{sys.executable}" -m pip install -U -q sentence-transformers accelerate

print("✅ Extra libraries for local models run have been installed.")

✅ Extra libraries for local models run have been installed.


## 3. Configuration

Set up the necessary configurations for Weaviate and Azure OpenAI. 

**Action Required**: You must create a `.env` file in the same directory as this notebook and add your Azure OpenAI credentials.

In [14]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
HF_API_TOKEN = os.environ["HUGGINGFACE_API_TOKEN"]

# Embeddingd model for local run.
# If you have access to Gemma (you logged in via huggingface-cli), use: "google/embeddinggemma-300m" (768 dimensions)
# If you don't have access or encounter errors, use the standard one: "all-MiniLM-L6-v2" (384 dimensions)
LOCAL_EMBEDDING_MODEL_NAME = "google/embeddinggemma-300m"
# LOCAL_EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2" # Uncomment if Gemma doesn't work


# Text generation model for local run.
LOCAL_LLM_MODEL_NAME = "google/gemma-3-1b-it"


# --- VECTOR DATABASE CONFIGURATION ---
WEAVIATE_CONTAINER_NAME = "simple-rag-weaviate"
WEAVIATE_IMAGE = "semitechnologies/weaviate:1.33.7"
WEAVIATE_HTTP_PORT = 8080
WEAVIATE_HTTP_PORT_EXTERNAL = 28080
WEAVIATE_GRPC_PORT = 50051

print("✅ Configuration loaded.")

✅ Configuration loaded.


In [5]:
from huggingface_hub import login

login(token=HF_API_TOKEN)
print("Successfully logged in to Hugging Face!")

/home/k.bostanbekov/miniconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully logged in to Hugging Face!


## 4. Data Generation

Here we generate 25 simple, factual notes across 5 different topics. Each note contains a specific detail (like a number, a name, or a technical term) to make it uniquely identifiable during the retrieval phase of our RAG experiment. This ensures we are testing the retrieval mechanism, not just the general knowledge of the LLM.

In [6]:
documents_data = [
    # Topic 1: The Atacama Desert
    {
        "title": "Rain in the Driest Place",
        "content": "The Atacama Desert in Chile is the driest nonpolar desert in the world. Some weather stations there have never received rain. However, in 2015, a rare weather event brought enough rainfall to cause the desert to bloom with flowers, an event that occurs only once every 5 to 7 years."
    },
    {
        "title": "Mars-like Soil",
        "content": "The soil in the Atacama Desert is often compared to that of Mars. NASA has used the desert as a testing ground for instruments intended for Mars missions, such as the Sample Analysis at Mars (SAM) instrument suite, due to its extreme aridity and soil composition."
    },
    {
        "title": "Ancient Mummies",
        "content": "The Chinchorro mummies, found in the Atacama Desert, are the world's oldest artificially mummified human remains. Dated as early as 5050 BC, they predate the more famous Egyptian mummies by thousands of years."
    },
    {
        "title": "Stargazing Paradise",
        "content": "With its high altitude, clear skies, and lack of light pollution, the Atacama Desert is one of the best places on Earth for astronomy. It hosts major observatories like the Atacama Large Millimeter Array (ALMA), which consists of 66 high-precision antennas."
    },
    {
        "title": "Nitrate Mining History",
        "content": "In the late 19th and early 20th centuries, the Atacama was a major source of sodium nitrate, also known as 'white gold', which was used as a fertilizer and in explosives. The abandoned Humberstone and Santa Laura Saltpeter Works are now a UNESCO World Heritage site."
    },

    # Topic 2: Tardigrades (Water Bears)
    {
        "title": "Extremophile Survivors",
        "content": "Tardigrades, also known as water bears, are microscopic invertebrates famous for their ability to survive extreme conditions. They can enter a state of suspended animation called cryptobiosis, withstanding temperatures from -272°C (-458°F) to 150°C (302°F)."
    },
    {
        "title": "Space Travelers",
        "content": "In 2007, tardigrades became the first animals to survive exposure to outer space. As part of the FOTON-M3 mission, they were exposed to the vacuum and radiation of space for 10 days and many were successfully reanimated back on Earth."
    },
    {
        "title": "Radiation Resistance",
        "content": "Tardigrades can withstand radiation doses hundreds of times higher than what would be lethal to humans. They possess a unique protein called 'Dsup' (Damage suppressor) which protects their DNA from radiation damage."
    },
    {
        "title": "Microscopic Size",
        "content": "Despite their resilience, adult tardigrades are typically only about 0.5 mm long. They have eight legs, each tipped with four to eight claws, and are found in diverse environments from mountaintops to the deep sea."
    },
    {
        "title": "Anhydrobiosis",
        "content": "A key survival mechanism for tardigrades is anhydrobiosis, the ability to survive extreme dehydration. In this state, they can reduce their body's water content to as little as 3% of normal, replacing it with protective sugars like trehalose."
    },

    # Topic 3: The Magnus Effect
    {
        "title": "Physics of a Curveball",
        "content": "The Magnus effect is the physical phenomenon responsible for the curved path of a spinning object moving through a fluid (like air). A spinning baseball creates a pressure differential, with higher pressure on the side spinning against the airflow, causing it to curve."
    },
    {
        "title": "Flettner Rotors",
        "content": "In the 1920s, German engineer Anton Flettner invented the Flettner rotor, a smooth cylinder that uses the Magnus effect for propulsion. The ship Buckau, later renamed Baden Baden, successfully crossed the Atlantic in 1926 using two of these large rotating cylinders as sails."
    },
    {
        "title": "Application in Sports",
        "content": "Besides baseball, the Magnus effect is crucial in many sports. It explains the 'topspin' in tennis that makes the ball dip sharply, the 'curl' on a soccer free-kick, and the 'hook' or 'slice' in golf. The effect's strength depends on the spin rate and speed of the ball."
    },
    {
        "title": "Reverse Magnus Effect",
        "content": "Under specific conditions, typically with a smooth ball at a certain range of speeds known as the 'drag crisis', a reverse Magnus effect can occur. This causes the ball to curve in the opposite direction to what is normally expected, a phenomenon observed in early experiments with smooth cylinders."
    },
    {
        "title": "Bernoulli's Principle",
        "content": "The Magnus effect is an application of Bernoulli's principle. The side of the spinning ball moving in the same direction as the airflow experiences faster air speed and lower pressure, while the opposite side has slower air and higher pressure, creating a net force."
    },

    # Topic 4: The Great Emu War
    {
        "title": "Australia's Unusual Conflict",
        "content": "The Great Emu War was a nuisance wildlife management military operation undertaken in Australia in 1932. Soldiers armed with machine guns were dispatched to combat a population of over 20,000 emus that were ravaging crops in Western Australia."
    },
    {
        "title": "Military Hardware",
        "content": "The military force, led by Major G.P.W. Meredith of the Royal Australian Artillery, was equipped with two Lewis automatic machine guns and 10,000 rounds of ammunition. The operation was seen as a way to assist struggling post-WWI veteran farmers."
    },
    {
        "title": "The Elusive Emus",
        "content": "The emus proved to be surprisingly difficult targets. They would scatter into small groups when fired upon, and their tough feathers provided some protection. After one week, only about 50 emus had been killed using approximately 2,500 rounds."
    },
    {
        "title": "Operation Outcome",
        "content": "The operation was widely seen as a failure and a source of public ridicule. A second attempt was made, which was slightly more successful, but ultimately the military withdrew. The conflict humorously cemented the emu's reputation as a resilient national symbol."
    },
    {
        "title": "A Second Attempt",
        "content": "Following the initial failure, a second military attempt was launched in mid-November 1932. This effort was more effective, with reports claiming nearly 1,000 emus were killed. However, the government ultimately decided that military intervention was not a cost-effective solution."
    },

    # Topic 5: The Baader-Meinhof Phenomenon
    {
        "title": "Frequency Illusion",
        "content": "The Baader-Meinhof phenomenon, also known as the frequency illusion, is a cognitive bias where, after noticing something for the first time, there is a tendency to notice it more often, leading one to believe that it has an increased frequency of occurrence."
    },
    {
        "title": "Two Cognitive Processes",
        "content": "This illusion is caused by two psychological processes working together. The first is selective attention, where your brain subconsciously looks for the new thing. The second is confirmation bias, where each new sighting reinforces your belief that the thing is ubiquitous."
    },
    {
        "title": "Origin of the Name",
        "content": "The name originated in 1994 when a commenter on the St. Paul Pioneer Press online board mentioned hearing about the German Baader-Meinhof Gang twice in 24 hours. This led to others sharing similar experiences, and the name stuck, despite having no real connection to the gang itself."
    },
    {
        "title": "Not a Real Increase",
        "content": "It's important to understand that the frequency of the event or item is not actually increasing. The phenomenon is purely a matter of perception, where your brain is newly primed to notice what was always there. It's a quirk of how our pattern-recognition systems work."
    },
    {
        "title": "Examples in Daily Life",
        "content": "A common example of the Baader-Meinhof phenomenon is when you buy a new car, and suddenly you start seeing the same model everywhere. Another is learning a new word and then hearing it multiple times in the following days. This is not synchronicity, but your brain's heightened awareness."
    }
]

print(f"✅ Generated {len(documents_data)} documents across 5 topics.")

✅ Generated 25 documents across 5 topics.


## 5. Docker Environment Setup

We will now start the Weaviate database using a Docker container. The following cells will check for Docker, pull the required image, and run the container with the correct port mappings.

In [ ]:
# First, ensure no old container with the same name is running
print(f"--- Stopping and removing any existing container named '{WEAVIATE_CONTAINER_NAME}' ---")
stop_command = f"docker stop {WEAVIATE_CONTAINER_NAME} 2>/dev/null; docker rm {WEAVIATE_CONTAINER_NAME} 2>/dev/null"
run_shell_command(stop_command)
print("Cleanup complete.")

# Now, run the new Weaviate container
print(f"\n--- Starting Weaviate container '{WEAVIATE_CONTAINER_NAME}' ---")
run_command = (
    f"docker run -d "
    f"--name {WEAVIATE_CONTAINER_NAME} "
    f"-p {WEAVIATE_HTTP_PORT_EXTERNAL}:{WEAVIATE_HTTP_PORT} "
    f"-p {WEAVIATE_GRPC_PORT}:{WEAVIATE_GRPC_PORT} "
    f"-e AUTHENTICATION_ANONYMOUS_ACCESS_ENABLED=true "
    f"-e PERSISTENCE_DATA_PATH=/var/lib/weaviate "
    f"-e DEFAULT_VECTORIZER_MODULE=none "
    f"-e ENABLE_MODULES='' "
    f"-e CLUSTER_HOSTNAME=node1 "
    f"{WEAVIATE_IMAGE}"
)

result = run_shell_command(run_command)

if result["success"]:
    print("✅ Weaviate container started successfully.")
    print("Waiting a few seconds for the service to initialize...")
    import time
    time.sleep(10) # Give Weaviate time to start up
else:
    print("❌ Failed to start Weaviate container.")
    print(f"Stderr: {result['stderr']}")

# Display container statistics
print("\n--- Weaviate Container Stats ---")
stats_result = run_shell_command(f"docker stats {WEAVIATE_CONTAINER_NAME} --no-stream")
print(stats_result["stdout"])
if stats_result["stderr"]:
    print(f"Stderr: {stats_result['stderr']}")

--- Stopping and removing any existing container named 'simple-rag-weaviate' ---
Cleanup complete.

--- Starting Weaviate container 'simple-rag-weaviate' ---
✅ Weaviate container started successfully.
Waiting a few seconds for the service to initialize...

--- Weaviate Container Stats ---
CONTAINER ID   NAME                  CPU %     MEM USAGE / LIMIT     MEM %     NET I/O           BLOCK I/O    PIDS
b506948f2ca8   simple-rag-weaviate   0.52%     39.85MiB / 187.4GiB   0.02%     11.3kB / 3.67kB   0B / 201kB   40


## 6. Embeddings and Data Ingestion

In this section, we will:
1.  Set up the LangChain clients for Azure OpenAI (for both embeddings and chat).
2.  Generate vector embeddings for each of our 25 documents.
3.  Connect to our Weaviate instance.
4.  Define a data schema (a "collection") in Weaviate.
5.  Batch-insert all documents and their vectors into the collection.

**NOTE**: The loacal models examples have beed added as alternative, 
make sense to split this cell on two: clietns and data ingestion

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.runnables import Runnable, RunnableConfig
import weaviate
import weaviate.classes as wvc
from weaviate.util import generate_uuid5
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import torch
import numpy as np

# --- Wrapper class for the local Embeddings model ---
class LocalHuggingFaceEmbeddings:
    """
    This class adapts a local SentenceTransformer model
    to the LangChain interface, which expects the methods embed_documents and embed_query.
    """
    def __init__(self, model_name):
        print(f"📥 Loading local embedding model: {model_name}...")
        try:
            self.model = SentenceTransformer(model_name, device="cuda" if torch.cuda.is_available() else "cpu")
            print("✅ Local embedding model loaded successfully. Device: " + ("cuda" if torch.cuda.is_available() else "cpu"))
        except Exception as e:
            print(f"❌ Error loading {model_name}. Falling back to 'all-MiniLM-L6-v2'.")
            print(f"Error details: {e}")
            self.model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    def embed_documents(self, texts):
        # Returns a list of lists
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        return embeddings.tolist()

    def embed_query(self, text):
        # Returns a single list
        embedding = self.model.encode(text, convert_to_numpy=True)
        return embedding.tolist()


# --- Wrapper class for the local LLM ---
class LocalHuggingFaceChatModel(Runnable):
    """
    A simple wrapper around the Transformers Pipeline to make it compatible
    with LangChain's 'invoke' method and the pipe '|' operator.
    """
    def __init__(self, model_name):
        print(f"📥 Loading local LLM: {model_name}...")
        # This is the 'Automatic Transmission' setup we discussed:
        # 1. device=-1 forces CPU usage.
        # 2. torch_dtype=torch.float32 is the fastest format for CPU.
        self.pipe = pipeline(
            "text-generation",
            model=model_name,
            device="cuda" if torch.cuda.is_available() else "cpu",
            torch_dtype=torch.float32
        )
        print("✅ Local LLM loaded successfully.")

    def invoke(self, input_data, config: RunnableConfig = None, **kwargs):
        """
        Adapts LangChain inputs (PromptValue or Messages) to the pipeline format.
        """
        # 1. Convert LangChain input to the list-of-dicts format expected by the pipeline
        messages = []

        # Handle LangChain PromptValue (which has .to_messages())
        if hasattr(input_data, 'to_messages'):
            lc_messages = input_data.to_messages()
            for msg in lc_messages:
                # Map LangChain message types to role strings
                role = "user"
                if msg.type == "system": role = "system"
                elif msg.type == "ai": role = "assistant"

                # Gemma pipeline expects content as a list of dicts or string.
                messages.append({"role": role, "content": [{"type": "text", "text": msg.content}]})

        # Handle raw string input (fallback)
        elif isinstance(input_data, str):
            messages = [{"role": "user", "content": [{"type": "text", "text": input_data}]}]

        # 2. Run the pipeline ("Automatic Transmission")
        # We set max_new_tokens to limit the answer length
        outputs = self.pipe(messages, max_new_tokens=512)

        # 3. Extract the generated text
        # The pipeline returns a list of dicts. The last message is the assistant's reply.
        generated_text = outputs[0]['generated_text'][-1]['content']

        # 4. Return as an AIMessage to satisfy LangChain's StrOutputParser
        return AIMessage(content=generated_text)

In [23]:
# --- 1. Setup LangChain Clients ---
print("--- 1. Setting up AI clients ---")
try:
    # Embedding Model Setup
    embeddings_model = LocalHuggingFaceEmbeddings(LOCAL_EMBEDDING_MODEL_NAME)

    # Chat Model Setup
    chat_model = LocalHuggingFaceChatModel(LOCAL_LLM_MODEL_NAME)
    print("✅ AI clients initialized.")

except Exception as e:
    print(f"❌ Failed to initialize AI clients. Please check your .env file or model names. Error: {e}")
    # Stop execution if clients fail to initialize
    raise

--- 1. Setting up AI clients ---
📥 Loading local embedding model: google/embeddinggemma-300m...
✅ Local embedding model loaded successfully.Device: cuda
📥 Loading local LLM: google/gemma-3-1b-it...


Device set to use cuda


✅ Local LLM loaded successfully.
✅ AI clients initialized.


In [24]:
# --- 2. Generate Embeddings ---
print("\n--- 2. Generating embeddings for all documents ---")
contents_to_embed = [doc['content'] for doc in documents_data]
vector_embeddings = embeddings_model.embed_documents(contents_to_embed)
print(f"✅ Generated {len(vector_embeddings)} embeddings. Vector dimension: {len(vector_embeddings[0])}")

# Add embeddings to our data
for i, doc in enumerate(documents_data):
    doc['content_vector'] = vector_embeddings[i]


--- 2. Generating embeddings for all documents ---
✅ Generated 25 embeddings. Vector dimension: 768


In [25]:
# --- 3. Connect to Weaviate ---
print("\n--- 3. Connecting to Weaviate ---")
weaviate_client = weaviate.connect_to_local(
    host="localhost",
    port=WEAVIATE_HTTP_PORT_EXTERNAL,
    grpc_port=WEAVIATE_GRPC_PORT
)
if weaviate_client.is_ready():
    print("✅ Successfully connected to Weaviate.")
else:
    print("❌ Failed to connect to Weaviate.")
    weaviate_client.close()
    raise ConnectionError("Could not connect to Weaviate instance.")


--- 3. Connecting to Weaviate ---
✅ Successfully connected to Weaviate.


In [ ]:
# --- 4. Define and Create Weaviate Collection ---
COLLECTION_NAME = "SimpleRAG"
print(f"\n--- 4. Creating Weaviate collection: '{COLLECTION_NAME}' ---")

# Delete collection if it already exists for a clean run
if weaviate_client.collections.exists(COLLECTION_NAME):
    weaviate_client.collections.delete(COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'.")

# Create new DB schema for our documents
rag_collection = weaviate_client.collections.create(
    name=COLLECTION_NAME,
    properties=[
        wvc.config.Property(name="title", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="content", data_type=wvc.config.DataType.TEXT),
    ],
    vector_config=wvc.config.Configure.Vectors.self_provided(
        vector_index_config=wvc.config.Configure.VectorIndex.hnsw(
            distance_metric=wvc.config.VectorDistances.COSINE
        )
    )
)
print(f"✅ Collection '{COLLECTION_NAME}' created successfully.")


--- 4. Creating Weaviate collection: 'SimpleRAG' ---
Deleted existing collection 'SimpleRAG'.
✅ Collection 'SimpleRAG' created successfully.


In [27]:
# --- 5. Batch-Insert Data ---
print(f"\n--- 5. Ingesting {len(documents_data)} documents into Weaviate ---")

# Use a context manager to automatically handle batching
with rag_collection.batch.dynamic() as batch:
    for doc in documents_data:
        properties = {
            "title": doc["title"],
            "content": doc["content"]
        }
        batch.add_object(
            properties=properties,
            vector=doc["content_vector"],  # Use default vector
            uuid=generate_uuid5(doc["title"])  # Generate a consistent UUID based on the title
        )

print(f"✅ Data ingestion complete. Total objects in collection: {len(rag_collection)}")

# Close the client connection
weaviate_client.close()


--- 5. Ingesting 25 documents into Weaviate ---
✅ Data ingestion complete. Total objects in collection: 25


## 7. RAG Experiment

Now we perform the core RAG experiment. For each of our five topics, we will ask a question and follow the RAG pipeline to generate an answer.

**The Pipeline:**
1.  **Expand Query**: Use an LLM to rephrase the user's simple question into a richer, more descriptive query.
2.  **Embed Query**: Generate a vector embedding for the expanded query.
3.  **Retrieve Documents**: Search Weaviate for the top 5 documents most similar to the query vector.
4.  **Generate Answer**: Pass the retrieved documents as context to another LLM call and ask it to synthesize a final, bulleted answer based *only* on the provided information.

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import pandas as pd
from IPython.display import display, Markdown

# Re-connect to Weaviate for the experiment
weaviate_client = weaviate.connect_to_local(
    host="localhost",
    port=WEAVIATE_HTTP_PORT_EXTERNAL,
    grpc_port=WEAVIATE_GRPC_PORT
)
rag_collection = weaviate_client.collections.get(COLLECTION_NAME)

# --- Define Test Questions ---
user_query = 'What do you know about the ALMA observatory?'

# 1. Chain for Query Expansion
expansion_prompt = ChatPromptTemplate.from_template(
    "You are an expert in information retrieval. "
    "Please rephrase the following user query to be more descriptive and detailed, "
    "making it suitable for a vector database search. "
    "Return only the rephrased query, without any additional text, headers, or explanations. "
    "\n\nOriginal Query: '{query}'\n\nRephrased Query:"
)
query_expansion_chain = expansion_prompt | chat_model | StrOutputParser()

# 2. Chain for Final Answer Generation (with RAG context)
generation_prompt = ChatPromptTemplate.from_template(
    "You are a factual assistant. "
    "Your task is to answer the user's question based only on the provided context, "
    "do not use common knowledge, do not correct mistakes in provided context. "
    "Synthesize the information from the context into a concise, bullet-point summary. "
    "Focus on specific details like names, numbers, and technical terms mentioned in the context. "
    "If the context does not contain the information needed to answer the question, "
    "you must state: 'The provided context does not contain the answer to this question.' "
    "\n\nContext:\n{context}\n\nQuestion: {question}"
)
answer_generation_chain = generation_prompt | chat_model | StrOutputParser()


# --- Run the Experiment ---

experiment_results = []

display(Markdown(f"### Original Question: {user_query}"))

# 1. Expand the query
expanded_query = query_expansion_chain.invoke({"query": user_query})
display(Markdown(f"**Rephrased Query for Search:** {expanded_query}"))

# --- RAG Pipeline ---

# Embed Expanded Query
query_embedding = embeddings_model.embed_query(expanded_query)

# Retrieve Documents from Weaviate
retrieved_objects = rag_collection.query.near_vector(
    near_vector=query_embedding,
    limit=5,
    return_metadata=wvc.query.MetadataQuery(distance=True)
)

retrieved_docs_content = [obj.properties['content'] for obj in retrieved_objects.objects]
context_for_llm = "\n\n---\n\n".join(retrieved_docs_content)


# 3. Generate Final Answer using RAG
final_answer = answer_generation_chain.invoke({
    "context": context_for_llm,
    "question": user_query
})
display(Markdown(f"**Answer to the original query, with RAG:**\n{final_answer}"))

# Close the client connection
weaviate_client.close()

### Original Question: What do you know about the ALMA observatory?

**Rephrased Query for Search:** Please provide a detailed search query focusing on specific aspects of the ALMA observatory, including its location, scientific goals, data types, operational history, and any notable findings or challenges.


**Answer to the original query, with RAG:**
Here’s a bullet-point summary of what I know about the ALMA observatory:

*   It’s a major observatory located in the Atacama Desert.
*   It hosts 66 high-precision antennas.
*   NASA used the desert as a testing ground for instruments intended for Mars missions, including the Sample Analysis at Mars (SAM) instrument suite.
*   The soil in the Atacama Desert is compared to that of Mars.